# Splink on Snowflake: Zero-JAR Approach with RapidFuzz

This notebook demonstrates a JAR-free approach to running [Splink v4](https://github.com/moj-analytical-services/splink)
on Snowflake via Snowpark Connect for Spark (SCOS). Instead of uploading Java UDF JARs,
we register Python scalar UDFs backed by [RapidFuzz](https://github.com/maxbachmann/RapidFuzz)
(C++ engine, available on Snowflake's Anaconda channel).

**What you'll learn:**
- How to register Python UDFs with package dependencies on SCOS
- How to write custom Splink comparison definitions that reference your UDFs
- How to run the full pipeline without any JAR uploads

**Prerequisites:**
- Run `00_setup.sql` first to create the relevant resources (Database, Compute Pool, Warehouse, etc.)
- Upload the accompanying `pyproject.toml` into the same directory as this notebook

**Comparisons used:**
- Custom `ForenameSurnameComparison` equivalent (first_name + surname) — uses `jw_rapidfuzz`
- Custom `NameComparison` equivalent (full_name) — uses `jw_rapidfuzz`
- Custom `DateOfBirthComparison` equivalent (dob) — uses `dl_rapidfuzz` + native date arithmetic
- `PostcodeComparison` (postcode_fake) — native (no change)
- `LevenshteinAtThresholds` (occupation) — native (no change)
- `ExactMatch` (gender) — native (no change)
- `ExactMatch` (birth_place) with TF — native (no change)

**Dataset:** `splink_datasets.historical_50k` (50,000 rows)  
**Warehouse:** `SPLINK_WH` (MEDIUM)

In [ ]:
%pip install uv

In [ ]:
!uv pip install --system -r pyproject.toml

In [ ]:
import pyspark
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.connect.functions import udf
from snowflake.snowpark_connect import init_spark_session
from snowflake.snowpark_connect.snowflake_session import SnowflakeSession
from splink import Linker, SparkAPI, SettingsCreator, block_on, splink_datasets
import splink.comparison_library as cl

## Connect to Snowflake

Initialize a Spark session backed by Snowflake via Snowpark Connect and set the relevant Spark and Snowpark Connect configurations, followed by your Snowflake context.

In [ ]:
conf = pyspark.SparkConf()
conf.set("spark.sql.session.timeZone", "UTC")
conf.set("spark.sql.ansi.enabled", "false")
conf.set("snowpark.connect.cte.optimization_enabled", "true")
conf.set("snowpark.connect.integralTypesEmulation", "enabled")

spark = init_spark_session(app_name="splink", conf=conf)

In [ ]:
WAREHOUSE = "SPLINK_WH"
DATABASE = "SPLINK_DEMO"
SCHEMA = "PUBLIC"
ROLE = "SPLINK_ROLE"
NUM_PARTITIONS = 8

session = SnowflakeSession(spark)
session.sql(f"USE ROLE {ROLE}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Session ready. num_partitions_on_repartition={NUM_PARTITIONS}")
session.sql("SELECT CURRENT_WAREHOUSE() AS wh, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS sch").show()

## Register RapidFuzz UDFs

Instead of uploading Java JARs, we register Python scalar UDFs backed by RapidFuzz's C++
engine. The `rapidfuzz` package is resolved from Snowflake's Anaconda channel at execution time —
no `addArtifact` calls needed.

In [ ]:
spark.conf.set("snowpark.connect.udf.packages", "[rapidfuzz]")

@udf(returnType=DoubleType(), useArrow=True)
def jw_rapidfuzz(s1: str, s2: str) -> float:
    from rapidfuzz.distance import JaroWinkler
    if s1 is None or s2 is None:
        return None
    return JaroWinkler.similarity(s1, s2)

@udf(returnType=IntegerType(), useArrow=True)
def dl_rapidfuzz(s1: str, s2: str) -> int:
    from rapidfuzz.distance import DamerauLevenshtein
    if s1 is None or s2 is None:
        return None
    return DamerauLevenshtein.distance(s1, s2)

spark.udf.register("jw_rapidfuzz", jw_rapidfuzz)
spark.udf.register("dl_rapidfuzz", dl_rapidfuzz)
print("Registered: jw_rapidfuzz (JW similarity 0-1), dl_rapidfuzz (DL distance integer)")

## Custom Splink Comparisons

Splink's built-in `ForenameSurnameComparison`, `NameComparison`, and `DateOfBirthComparison`
reference Java UDF names internally. To use our Python UDFs instead, we define equivalent
comparison dictionaries that call `jw_rapidfuzz` and `dl_rapidfuzz` by name in SQL expressions.

In [ ]:
JW = "jw_rapidfuzz"
DL = "dl_rapidfuzz"

def custom_forename_surname_comparison(fn_col, sn_col, jw_fn=JW):
    """Mirrors ForenameSurnameComparison: columns-reversed check + JW at thresholds."""
    return {
        "comparison_levels": [
            {"sql_condition": f"`{fn_col}_l` IS NULL OR `{fn_col}_r` IS NULL OR `{sn_col}_l` IS NULL OR `{sn_col}_r` IS NULL",
             "label_for_charts": "Null", "is_null_level": True},
            {"sql_condition": f"`{fn_col}_l` = `{fn_col}_r` AND `{sn_col}_l` = `{sn_col}_r`",
             "label_for_charts": "Exact match on both"},
            {"sql_condition": f"`{fn_col}_l` = `{sn_col}_r` AND `{fn_col}_r` = `{sn_col}_l`",
             "label_for_charts": "Columns reversed (forename/surname swap)"},
            {"sql_condition": f"{jw_fn}(`{fn_col}_l`, `{fn_col}_r`) >= 0.92 AND {jw_fn}(`{sn_col}_l`, `{sn_col}_r`) >= 0.92",
             "label_for_charts": "JW >= 0.92 on both"},
            {"sql_condition": f"{jw_fn}(`{fn_col}_l`, `{fn_col}_r`) >= 0.88 AND {jw_fn}(`{sn_col}_l`, `{sn_col}_r`) >= 0.88",
             "label_for_charts": "JW >= 0.88 on both"},
            {"sql_condition": f"`{sn_col}_l` = `{sn_col}_r`",
             "label_for_charts": "Exact match on surname only"},
            {"sql_condition": f"`{fn_col}_l` = `{fn_col}_r`",
             "label_for_charts": "Exact match on forename only"},
            {"sql_condition": "ELSE", "label_for_charts": "All other"},
        ],
        "comparison_description": f"ForenameSurnameComparison ({fn_col} + {sn_col}) via {jw_fn}",
    }

def custom_name_comparison(col, jw_fn=JW):
    """Mirrors NameComparison: exact + JW at 0.92/0.88/0.7."""
    return {
        "comparison_levels": [
            {"sql_condition": f"`{col}_l` IS NULL OR `{col}_r` IS NULL",
             "label_for_charts": f"{col} is NULL", "is_null_level": True},
            {"sql_condition": f"`{col}_l` = `{col}_r`",
             "label_for_charts": f"Exact match on {col}"},
            {"sql_condition": f"{jw_fn}(`{col}_l`, `{col}_r`) >= 0.92",
             "label_for_charts": f"JW >= 0.92"},
            {"sql_condition": f"{jw_fn}(`{col}_l`, `{col}_r`) >= 0.88",
             "label_for_charts": f"JW >= 0.88"},
            {"sql_condition": f"{jw_fn}(`{col}_l`, `{col}_r`) >= 0.7",
             "label_for_charts": f"JW >= 0.7"},
            {"sql_condition": "ELSE", "label_for_charts": "All other"},
        ],
        "comparison_description": f"NameComparison on {col} via {jw_fn}",
    }

def custom_dob_comparison(col="dob", dl_fn=DL):
    """Mirrors DateOfBirthComparison: DL + date arithmetic levels."""
    return {
        "comparison_levels": [
            {"sql_condition": f"`{col}_l` IS NULL OR `{col}_r` IS NULL",
             "label_for_charts": "dob is NULL", "is_null_level": True},
            {"sql_condition": f"`{col}_l` = `{col}_r`",
             "label_for_charts": "Exact match"},
            {"sql_condition": f"{dl_fn}(`{col}_l`, `{col}_r`) <= 1",
             "label_for_charts": "DL distance <= 1"},
            {"sql_condition": f"ABS(UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_l`, 'yyyy-MM-dd'))) - UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_r`, 'yyyy-MM-dd')))) <= 2629800.0",
             "label_for_charts": "Date diff <= 1 month"},
            {"sql_condition": f"ABS(UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_l`, 'yyyy-MM-dd'))) - UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_r`, 'yyyy-MM-dd')))) <= 31557600.0",
             "label_for_charts": "Date diff <= 1 year"},
            {"sql_condition": f"ABS(UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_l`, 'yyyy-MM-dd'))) - UNIX_TIMESTAMP(date(try_to_timestamp(`{col}_r`, 'yyyy-MM-dd')))) <= 315576000.0",
             "label_for_charts": "Date diff <= 10 years"},
            {"sql_condition": "ELSE", "label_for_charts": "All other"},
        ],
        "comparison_description": f"DateOfBirthComparison on {col} via {dl_fn}",
    }

print("Custom comparison builders defined.")

## Load Data

Splink ships with a built-in synthetic dataset (`historical_50k`) designed for
record linkage testing. We load it into a Spark DataFrame backed by Snowflake.

In [ ]:
pdf = splink_datasets.historical_50k
df = spark.createDataFrame(pdf)
row_count = df.count()
print(f"Loaded historical_50k: {row_count:,} rows")

## Configure the Linker

Wire the custom comparison definitions into Splink's `SettingsCreator` alongside the
native comparisons that don't need JARs. Note `register_udfs_automatically=False` since
we've already registered our own Python UDFs above.

In [ ]:
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method="persist",
    register_udfs_automatically=False,
    num_partitions_on_repartition=NUM_PARTITIONS,
)

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        custom_forename_surname_comparison("first_name", "surname"),
        custom_name_comparison("full_name"),
        custom_dob_comparison("dob"),
        cl.PostcodeComparison("postcode_fake"),
        cl.LevenshteinAtThresholds("occupation", [2, 4]),
        cl.ExactMatch("gender"),
        cl.ExactMatch("birth_place").configure(term_frequency_adjustments=True),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("first_name", "dob"),
        block_on("surname", "birth_place"),
        block_on("postcode_fake"),
    ],
    retain_intermediate_calculation_columns=True,
    em_convergence=0.001,
    max_iterations=30,
)

linker = Linker(df, settings, db_api=db_api)
print("Linker constructed. Zero JARs — Python UDFs only.")

## Train the Model

Splink's training consists of four steps:
1. Estimate the probability that two random records match
2. Estimate u-values (agreement rates among non-matches) via random sampling
3. EM pass blocking on (first_name, surname)
4. EM pass blocking on (dob)

In [ ]:
deterministic_rules = [
    block_on("first_name", "surname", "dob"),
    block_on("surname", "birth_place"),
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=1e8, seed=42)
linker.training.estimate_parameters_using_expectation_maximisation(block_on("first_name", "surname"))
linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))
print("Training complete.")

## Predict and Cluster

Run inference to find record pairs with match probability >= 0.85, then cluster
those pairs into entity groups at a stricter 0.95 threshold.

In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.85)
n_pairs = df_predict.as_spark_dataframe().count()
print(f"Predicted pairs (p >= 0.85): {n_pairs:,}")

df_clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95
)
clusters_sdf = df_clusters.as_spark_dataframe()
n_clusters = clusters_sdf.select("cluster_id").distinct().count()
print(f"Distinct clusters (p >= 0.95): {n_clusters:,}")